# Piano B — Notebook A (Account 1): baseline e TSception

Coda, in ordine di priorità: baseline su feature (CPU, con 100 permutazioni) → Optuna TSception (60 trial)
ed EEGNet (40) → TSception/EEGNet con EA su split dei foundation model e leave-clip-out (5 seed), incluso
BN-α offline/online → ablazioni senza EA → ablazione EMG con TSception → permutazioni per il Δstim →
sonda della clip → valenza binaria.

Impostazioni: Accelerator **GPU T4 x2**, Internet off. Avvio: **Save Version → Save & Run All (Commit)**.
Per riprendere: aggiungi come input l'output della versione precedente di questo notebook e rilancia.

In [ ]:
# Setup: trova il codice (Dataset "pianob-code"), lo copia fuori da /kaggle/working (così non finisce
# negli output) e lo aggiunge a sys.path. Non serve modificare nulla qui.
import glob, os, shutil, sys, time
os.environ.setdefault("PIANOB_T0", str(time.time()))      # inizio della sessione (guardia delle 12 h)
os.environ.setdefault("PIANOB_BUDGET_H", "11.3")           # stop pulito prima del limite Kaggle
SYNTHETIC = os.environ.get("PIANOB_SYNTHETIC") == "1"      # solo per prove locali


def _find_code():
    cands = []
    roots = ["/kaggle/input"] if os.path.exists("/kaggle/input") else []
    roots += [os.getcwd(), os.path.dirname(os.getcwd())]
    for root in roots:
        for p in glob.glob(os.path.join(root, "**", "pianob", "__init__.py"), recursive=True):
            base = os.path.dirname(os.path.dirname(p))
            if "pianob_state" not in base:
                cands.append(base)
    assert cands, "Codice non trovato: aggiungi il Dataset 'pianob-code' come input (Add Input)."
    # se ci sono più copie, usa quella con VERSION più recente
    def _ver(b):
        v = os.path.join(b, "VERSION")
        return open(v).read().strip() if os.path.exists(v) else ""
    return sorted(cands, key=_ver)[-1]


CODE_SRC = _find_code()
if os.path.exists("/kaggle/working"):
    CODE = "/tmp/pianob_code"
    shutil.rmtree(CODE, ignore_errors=True)
    shutil.copytree(CODE_SRC, CODE, ignore=shutil.ignore_patterns("notebooks", "*.zip", "local_work"))
else:
    CODE = CODE_SRC
sys.path.insert(0, CODE)
os.chdir(CODE)
print("codice da", CODE_SRC, "| versione", open(os.path.join(CODE, "VERSION")).read().strip()
      if os.path.exists(os.path.join(CODE, "VERSION")) else "(locale)")

from pianob import kaggle_env, nb_helpers  # noqa: E402
nb_helpers.print_env()
kaggle_env.restore_state_from_inputs()

In [ ]:
ACCOUNT = "account1_nbA"
nb_helpers.run_queue("nbA_baselines_tsception", gpus="auto", synthetic=SYNTHETIC)
nb_helpers.queue_status("nbA_baselines_tsception")

## Riepilogo da incollare in chat
Copia la tabella e tutto il blocco tra `PIANOB_JSON_BEGIN` e `PIANOB_JSON_END`.
I file completi sono in `results_account1_nbA.zip` (scheda Output).

In [ ]:
nb_helpers.finish(ACCOUNT)